## Enviroment Setup & Project Paths
Set the project root and make the src package importable from this notebook.

In [2]:
# --- Notebook bootstrap (root, sys.path, masked prints) ---
import sys, os
from pathlib import Path

# Use your existing mask_path() if present; otherwise a tiny fallback
if "mask_path" not in globals():
    def mask_path(p: str | Path) -> str:
        p = Path(p)
        parts = p.parts
        # show only the last 3 segments, prefix with "..."
        return str(Path(*(["..."] + list(parts[-3:]))))

# Robust ROOT detection (works whether notebook lives in src/ or project root)
if "__file__" in globals():
    ROOT = Path(__file__).resolve().parents[1]
else:
    cwd = Path.cwd()
    ROOT = cwd.parent if cwd.name == "src" else cwd

SRC_DIR = ROOT / "src"

# Ensure both ROOT and src/ are importable
for path in (ROOT, SRC_DIR):
    s = str(path)
    if s not in sys.path:
        sys.path.insert(0, s)

# Project imports
import src.utils as u
import src.ade_client as ac

print("ROOT:", mask_path(ROOT))
print("SRC_DIR:", mask_path(SRC_DIR))
print("utils.to_jsonable:", hasattr(u, "to_jsonable"))
print("ade_client.parse_pdf:", hasattr(ac, "parse_pdf"))


ROOT: ...\Edilma Projects\LandingAI-Hack\coderisk-sf
SRC_DIR: ...\LandingAI-Hack\coderisk-sf\src
utils.to_jsonable: True
ade_client.parse_pdf: True


## Load Environment Variables and Initialize ADE Client
Load environment variables from the .env file and initialize the LandingAI ADE client using the API key.
The client will be used throughout the notebook to parse PDF documents via the ADE Parse API.

In [3]:
from dotenv import load_dotenv
from landingai_ade import LandingAIADE

# Load .env file from project root
env_path = ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f".env not found at {env_path}")
load_dotenv(env_path)

# Retrieve API key (support both variable names)
api_key = os.getenv("ADE_API_KEY") or os.getenv("VISION_AGENT_API_KEY")

if not api_key:
    raise ValueError(
        "LandingAI API key not found. "
        "Add ADE_API_KEY or VISION_AGENT_API_KEY to your .env file."
    )

# Initialize ADE client
ade_client = LandingAIADE(apikey=api_key)

print("ADE client initialized successfully!")

ADE client initialized successfully!


## Debug & Development Tools

In [4]:
# --- ADE Debug Helpers ---

def print_chunk_summary(parsed):
    chunks = parsed.get("chunks") or []
    print("Total chunks:", len(chunks))

    from collections import Counter
    types = [c.get("type") for c in chunks if isinstance(c, dict)]
    print("Chunk types:", Counter(types))

    # show first few chunks
    for i, ch in enumerate(chunks[:10]):
        print(f"[{i}] type={ch.get('type')} page={ch.get('page')} "
              f"len(markdown)={len((ch.get('markdown') or ''))}")

def save_debug_json(city, pdf, parsed, RESULTS_DIR):
    """Save the raw JSON response into results_folder/city/raw_json."""
    city_dir = RESULTS_DIR / city / "raw_json"
    city_dir.mkdir(parents=True, exist_ok=True)
    out = city_dir / f"{pdf.stem}_debug.json"

    import json
    out.write_text(json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved debug JSON to:", out)


In [ ]:
# --- Quick test of extraction (run after PDF discovery) ---
if 'city_pdfs' in globals() and city_pdfs:
    city, pdf = city_pdfs[0]   # first PDF only
    print("Testing:", city, pdf)

    parsed = ac.parse_pdf(pdf)
    print("Keys in parsed:", parsed.keys())

    df = ac.extract_cases_df(parsed)
    print("DataFrame shape:", df.shape)
    print(df.head(3))
else:
    print("Run PDF discovery first to populate city_pdfs variable")

## Basic Data Analysis 

In [ ]:
## Data cleaning for Boca Raton dataset

boca2 = boca.copy()

# remove obvious header repeats (rows whose first col literally equals the header name)
if "Main Address" in boca2.columns:
    mask_headers = boca2["Main Address"].astype(str).str.strip().eq("Main Address")
    boca2 = boca2[~mask_headers]

# drop the description-only rows that slipped under "Main Address"
mask_descr = boca2["Main Address"].astype(str).str.startswith("Description:", na=False)
boca2 = boca2[~mask_descr]

# optional: coerce date column if present
for c in ("Resolved Date","Opened Date","Closed Date","Compliance Date","Citation Issued"):
    if c in boca2.columns:
        boca2[c] = pd.to_datetime(boca2[c], errors="coerce")
        
boca2["city"] = "Boca Raton"
boca2["source_file"] = "boca_JustFOIA_Request_2024-9180-30p.pdf"

boca2.head(10)


In [ ]:
#address analysis
print("Unique addresses:", boca2["Main Address"].nunique() if "Main Address" in boca2 else 0)
print("Sample addresses:", boca2["Main Address"].dropna().head(5).tolist())

# If these exist for Boca:
for c in ["Violation","Violation Status","Citation Issued","Compliance Date","Resolved Date","Parcel"]:
    if c in boca2:
        print(c, "non-null:", boca2[c].notna().sum())


### Normalization 

In [ ]:
#normalization to common schema
normalized_cols = [
    "case_id_raw","address_raw","parcel_raw",
    "violation","violation_status","citation_issued",
    "compliance_date","resolved_date",
    "opened_date","closed_date","assigned_to","project","district",
    "violation_fee_total","city","source_file"
]

boca_norm = pd.DataFrame(index=boca2.index, columns=normalized_cols)
boca_norm["address_raw"] = boca2.get("Main Address")
boca_norm["resolved_date"] = boca2.get("Resolved Date")
boca_norm["city"] = "Boca Raton"
boca_norm["source_file"] = "boca_JustFOIA_Request_2024-9180-30p.pdf"

boca_norm.head(10)


## Advanced Pipeline Tools & Debugging
These cells contain advanced tools for processing all cities, debugging, and complete pipeline execution.

##  Validate Working Directories and Discover Input PDFs
This cell verifies the project’s input structure, discovers PDFs grouped by city, and prints a clear inventory.
Each city corresponds to a subfolder inside input_folder/, and the city name is inferred from the folder name.
This ensures that the extraction and normalization pipeline operates on well-organized inputs.

In [5]:
#cell 17
from pathlib import Path
from collections import Counter

# Base directories
INPUT_DIR = ROOT / "input_folder"
RESULTS_DIR = ROOT / "results_folder"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def normalize_city_name(folder_name: str) -> str:
    """Standardize folder names into clean city labels."""
    return (
        folder_name.replace("_", " ")
                   .replace("-", " ")
                   .strip()
                   .title()
    )

def discover_city_pdfs(input_root: Path):
    """
    Scan input_folder/<city>/ for PDFs.
    Returns a list of (city_name, pdf_path).
    """
    items = []
    if not input_root.exists():
        print(f"Input directory not found: {input_root}")
        return items

    for sub in sorted(p for p in input_root.iterdir() if p.is_dir()):
        city = normalize_city_name(sub.name)
        pdfs = sorted(sub.glob("*.pdf"))

        # Optional: create matching results folder automatically
        city_results = RESULTS_DIR / sub.name / "raw_json"
        city_results.mkdir(parents=True, exist_ok=True)

        for pdf in pdfs:
            items.append((city, pdf))

    return items


# Discover PDFs
city_pdfs = discover_city_pdfs(INPUT_DIR)

# Inventory summary
counts = Counter([city for city, _ in city_pdfs])
print(f"Found {len(city_pdfs)} PDFs across {len(counts)} cities in {mask_path(INPUT_DIR)}")


for city, n in counts.items():
    print(f" - {city:<20} {n} PDFs")

# Preview first few items
if city_pdfs:
    print("\nPreview:")
    for city, p in city_pdfs[:5]:
        print(f"   {city:<20} -> {p.name}")
else:
    print("No PDFs found.")


Found 4 PDFs across 4 cities in ...\LandingAI-Hack\coderisk-sf\input_folder
 - Bocaraton            1 PDFs
 - Oaklandpark          1 PDFs
 - Pompano              1 PDFs
 - Wiltonmanor          1 PDFs

Preview:
   Bocaraton            -> boca_JustFOIA_Request_2024-9180-30p.pdf
   Oaklandpark          -> oakland_public_request_2024-041_Code_Cases_Violation-30p.pdf
   Pompano              -> pompanoViolations_20240124-30p.pdf
   Wiltonmanor          -> wilton_Code_Violation_to_March_2024-30p.pdf


In [ ]:
# --- Path consistency check ---
print("=== Checking Path Consistency ===")

# Check what we have in results_folder
results_cities = [d.name for d in RESULTS_DIR.iterdir() if d.is_dir()]
print(f"Existing result directories: {results_cities}")

# Check what we discovered from input_folder
input_cities = [city for city, _ in city_pdfs]
input_folders = [pdf.parent.name for _, pdf in city_pdfs]
print(f"Discovered cities: {input_cities}")  
print(f"Actual folder names: {set(input_folders)}")

# Test the folder name extraction
if city_pdfs:
    city, pdf = city_pdfs[0]
    folder_name = pdf.parent.name
    print(f"Example: City='{city}' -> Folder='{folder_name}'")

In [ ]:
# --- Save results for boca PDF --- saving tokens, credit almost exhausted
import json
from pathlib import Path
from src.ade_client import parse_pdf, extract_cases_df

city, pdf = city_pdfs[0]
parsed = parse_pdf(pdf)

# Save raw JSON - use actual folder name for consistency
folder_name = pdf.parent.name  # This gets "bocaraton" (lowercase)
(raw_dir := (RESULTS_DIR / folder_name / "raw_json")).mkdir(parents=True, exist_ok=True)
(raw_dir / f"{pdf.stem}.json").write_text(
    json.dumps(parsed, ensure_ascii=False, indent=2), encoding="utf-8"
)

# Save per-file table CSV
df = extract_cases_df(parsed)
(tables_dir := (RESULTS_DIR / folder_name / "tables")).mkdir(parents=True, exist_ok=True)
df.to_csv(tables_dir / f"{pdf.stem}.csv", index=False)

print("Saved:", mask_path(raw_dir / f"{pdf.stem}.json"))
print("Saved:", mask_path(tables_dir / f"{pdf.stem}.csv"))

In [ ]:
# --- Sanity check (to see if the data is ready for analysis) ---
import pandas as pd
boca = pd.read_csv(RESULTS_DIR / "bocaraton" / "tables" / "boca_JustFOIA_Request_2024-9180-30p.csv")

print("Rows:", len(boca), "Cols:", list(boca.columns))
print(boca.head(10))
print(boca.isna().mean().sort_values(ascending=False).head(8))  # null rates

## Data Analysis and Normalization
Now that we have extracted the raw data, let's clean it up and normalize it for analysis.

## Extraction Pipeline (ADE SDK · Table-First Parsing · Resumable · CSV + Parquet Output)

This cell loads the project’s ADE parsing utilities and the per-city normalization functions.
The extraction pipeline works as follows:

1. ADE parses each PDF using a table-first approach (tables from chunks, then fallback to markdown).
2. Each city’s raw table is passed through its corresponding normalization layer.
3. Extracted records are saved as both CSV and Parquet for downstream analytics.
4. The process is resumable, meaning each PDF only needs to be parsed once — saved JSONs and tables can be reused without re-calling the API.
5. This module setup prepares the extraction loop for the next step.

In [ ]:
# --- Chunk audit ---
chs = parsed.get("chunks") or []
print("Total chunks:", len(chs))

from collections import Counter
tally = Counter((c.get("type") or type(c).__name__) for c in chs)
print("Chunk types:", tally)

# peek the first few chunk meta
for i, c in enumerate(chs[:8]):
    print(f"[{i}] type={c.get('type')} page={ (c.get('grounding') or {}).get('page') } "
          f"len(markdown)={len((c.get('markdown') or ''))}")


In [ ]:
import pandas as pd

# add project root to sys.path
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ADE parsing utilities
from src.ade_client import parse_pdf, extract_cases_df

# Normalization dispatcher (chooses correct normalizer for each city)
from src.normalizer_dispatch import pick_normalizer

print("Imports loaded. Normalization system ready.")



In [ ]:
# ---- Extraction via ADE SDK (tables -> DataFrame), resumable, with raw JSON & per-file tables ----
import json
import pandas as pd
from pathlib import Path

from src.ade_client import parse_pdf, extract_cases_df  # wrapper returns dict with "markdown" + "chunks"

COMBINED_CSV     = RESULTS_DIR / "ade_extracted_results.csv"
COMBINED_PARQUET = RESULTS_DIR / "ade_extracted_results.parquet"

def ensure_city_dirs(city: str, pdf_path: Path) -> tuple[Path, Path]:
    """Return (raw_json_dir, tables_dir) for a city; create if missing."""
    # Use the actual folder name from the PDF path for consistency
    folder_name = pdf_path.parent.name  # This gets the actual folder name (e.g., "bocaraton")
    city_root   = RESULTS_DIR / folder_name
    raw_json    = city_root / "raw_json"
    tables_dir  = city_root / "tables"
    raw_json.mkdir(parents=True, exist_ok=True)
    tables_dir.mkdir(parents=True, exist_ok=True)
    return raw_json, tables_dir

def save_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

# Resume support
if COMBINED_CSV.exists():
    combined = pd.read_csv(COMBINED_CSV)
    processed = set(zip(combined.get("city", []), combined.get("source_file", [])))
    frames = [combined]
    print(f"Resuming: {len(combined)} rows from {combined['source_file'].nunique()} files.")
else:
    processed = set()
    frames = []
    print("Starting fresh extraction.")

# Main loop
for city, pdf_path in city_pdfs:
    key = (city, pdf_path.name)
    if key in processed:
        continue

    try:
        # 1) Parse with ADE; wrapper returns a plain dict
        parsed = parse_pdf(pdf_path)

        # 2) Save raw JSON for auditability
        raw_json_dir, tables_dir = ensure_city_dirs(city, pdf_path)
        save_json(parsed, raw_json_dir / f"{pdf_path.stem}.json")

        # 3) Convert tables -> DataFrame (robust helper uses markdown->html->read_html)
        df_raw = extract_cases_df(parsed)
        if df_raw.empty:
            print(f"[skip] No table rows: {city} :: {pdf_path.name}")
            continue

        # 4) Attach metadata and write a per-file CSV
        df_raw["city"] = city
        df_raw["source_file"] = pdf_path.name
        df_raw.to_csv(tables_dir / f"{pdf_path.stem}.csv", index=False)

        # 5) Append and persist combined outputs incrementally
        frames.append(df_raw)
        combined = pd.concat(frames, ignore_index=True, sort=False)
        combined.to_csv(COMBINED_CSV, index=False)
        combined.to_parquet(COMBINED_PARQUET, index=False)

        print(f"[ok] {city:<12} :: {pdf_path.name} (+{len(df_raw)} rows, total {len(combined)})")

    except Exception as e:
        print(f"[error] {city} :: {pdf_path.name} -> {e}")

print("Extraction complete.")
print("Outputs:")
print(f" - {COMBINED_CSV.name}")
print(f" - {COMBINED_PARQUET.name}")

In [ ]:
# Ensure ROOT is set and project root is on sys.path as you had earlier
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import importlib
import src.utils as u
import src.ade_client as ac

print("has to_jsonable in utils?:", hasattr(u, "to_jsonable"))
importlib.reload(u)
importlib.reload(ac)

from src.ade_client import parse_pdf, extract_cases_df
print("ade_client loaded from:", ac.__file__)
